In [ ]:
# 1. Imports and settings
from pathlib import Path
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Rectangle
import tifffile
from scipy.ndimage import gaussian_filter1d, percentile_filter
from scipy.signal import find_peaks, peak_widths
from scipy.interpolate import PchipInterpolator
from IPython.display import display

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
KNOWN_DIR = Path("pics")
SENSOR_CSV = Path("imx477_response.csv")
PERCENTILE, WINDOW = 25, 61
WL_MIN, WL_MAX = 400., 700.
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# 2. Function definitions (run this cell before the analysis cells)
@lru_cache(maxsize=8)
def get_full_pic(pic):
    with tifffile.TiffFile(str(pic)) as tif:
        raw = np.asarray(tif.series[1].asarray())
    if raw.ndim != 2:
        raise ValueError(f"Expected 2D RAW mosaic, got {raw.shape}: {pic}")
    return raw

def get_band(pic, band_prop=5):
    img = get_full_pic(str(pic))
    h = img.shape[0]
    size = h // band_prop
    rows = img.sum(axis=1, dtype=np.float64)
    cumulative = np.r_[0., np.cumsum(rows)]
    y0 = int(np.argmax(cumulative[size:] - cumulative[:-size]))
    return y0, y0 + size

def vert_sum(pic):
    img = get_full_pic(str(pic))
    y0, y1 = get_band(pic)
    return img[y0:y1].sum(axis=0, dtype=np.float64)

def find_peaks_smoothed(profile):
    smooth = gaussian_filter1d(np.asarray(profile, dtype=float), sigma=2)
    peaks, _ = find_peaks(smooth, prominence=0.008 * np.ptp(smooth), distance=5)
    return smooth, peaks

def get_bayer_profiles(pic):
    img = get_full_pic(str(pic))
    y0, y1 = get_band(pic)
    y0, y1 = 2*(y0//2), 2*(y1//2)
    region = img[y0:y1].astype(np.float64)
    # BGGR, average vertically, preserve horizontal Bayer-block positions
    b = region[0::2, 0::2].mean(axis=0)
    g = (region[0::2, 1::2].mean(axis=0) +
         region[1::2, 0::2].mean(axis=0)) / 2
    r = region[1::2, 1::2].mean(axis=0)
    x = 2*np.arange(len(r)) + 0.5
    return x, np.column_stack([r, g, b])

def get_percentile_baseline(pic, percentile=PERCENTILE, window=WINDOW):
    if not 0 <= percentile <= 100 or window < 3 or window % 2 != 1:
        raise ValueError("Use percentile 0-100 and an odd window >= 3")
    x, profiles = get_bayer_profiles(pic)
    baseline = np.column_stack([
        percentile_filter(profiles[:, k], percentile=percentile,
                          size=window, mode="nearest") for k in range(3)
    ])
    return x, profiles, baseline

def get_rgb_percentile(pic, percentile=PERCENTILE, window=WINDOW):
    x, profiles, baseline = get_percentile_baseline(pic, percentile, window)
    smooth, peaks = find_peaks_smoothed(vert_sum(pic))
    widths, _, left, right = peak_widths(smooth, peaks, rel_height=0.5)
    corrected = profiles - baseline
    rows = []
    for peak, width, l, r in zip(peaks, widths, left, right):
        half = max(4, int(np.ceil(width/2)))
        mask = (x >= peak-half) & (x <= peak+half)
        rgb = corrected[mask].mean(axis=0) if mask.any() else np.full(3, np.nan)
        valid = bool(np.isfinite(rgb).all() and (rgb > 0).all())
        ratios = rgb/rgb.sum() if valid else np.full(3, np.nan)
        rows.append(dict(pixel=int(peak), width_px=float(width),
                         left_px=float(l), right_px=float(r),
                         R=rgb[0], G=rgb[1], B=rgb[2],
                         r=ratios[0], g=ratios[1], b=ratios[2], valid=valid))
    return pd.DataFrame(rows)

def load_sensor_response(path=SENSOR_CSV):
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing {path}. Place your digitized IMX477 CSV beside the notebook.")
    sensor = pd.read_csv(path).sort_values("wavelength_nm")
    wavelengths = sensor["wavelength_nm"].to_numpy(float)
    if wavelengths.min() > WL_MIN or wavelengths.max() < WL_MAX:
        raise ValueError("Sensor CSV must cover 400-700 nm")
    grid = np.linspace(WL_MIN, WL_MAX, 3000)
    responses = np.column_stack([
        PchipInterpolator(wavelengths, sensor[c].to_numpy(float))(grid)
        for c in ("R", "G", "B")
    ])
    responses = np.clip(responses, 0, None)
    totals = responses.sum(axis=1, keepdims=True)
    if np.any(totals <= 0):
        raise ValueError("Zero RGB response in calibration range")
    return grid, responses/totals

wl_grid, rgb_grid = load_sensor_response()

def predict_sensor_wavelength(data):
    rgb = data[["r", "g", "b"]].to_numpy(float)
    distances = np.linalg.norm(rgb[:, None, :] - rgb_grid[None, :, :], axis=2)
    nearest = np.argmin(distances, axis=1)
    return wl_grid[nearest], distances[np.arange(len(nearest)), nearest]

def predict(pic):
    data = get_rgb_percentile(pic)
    data = data[data["valid"]].copy()
    if not data.empty:
        data["estimated_nm"], data["colour_distance"] = predict_sensor_wavelength(data)
    else:
        data["estimated_nm"] = pd.Series(dtype=float)
        data["colour_distance"] = pd.Series(dtype=float)
    return data

def wavelength_to_rgb(w):
    w = float(w)
    if not 380 <= w <= 780: return (0., 0., 0.)
    if w < 440: rgb = (-(w-440)/60, 0, 1)
    elif w < 490: rgb = (0, (w-440)/50, 1)
    elif w < 510: rgb = (0, 1, -(w-510)/20)
    elif w < 580: rgb = ((w-510)/70, 1, 0)
    elif w < 645: rgb = (1, -(w-645)/65, 0)
    else: rgb = (1, 0, 0)
    factor = (0.3+0.7*(w-380)/40 if w < 420 else
              0.3+0.7*(780-w)/80 if w > 700 else 1.)
    return tuple((max(0., c)*factor)**0.8 for c in rgb)

def fit_axis(data, reference_map=None):
    # Reference-based dispersion is only used for known lamps.
    # Unknown lamps use an approximate line fitted to RGB estimates.
    d = data.copy()
    if reference_map is not None:
        d["fit_nm"] = d["pixel"].map(reference_map)
        d = d.dropna(subset=["fit_nm"])
        source = "reference lines"
    else:
        d = d.dropna(subset=["estimated_nm"])
        d = d[d["colour_distance"] <= 0.15].copy()
        d["fit_nm"] = d["estimated_nm"]
        source = "RGB-estimated peaks (approximate)"
    if len(d) < 2:
        return None, source
    slope, intercept = np.polyfit(d["pixel"], d["fit_nm"], 1)
    return (float(slope), float(intercept)), source

def add_reference_lines(ax, refs, dark=False):
    for wavelength in refs:
        line = ax.axvline(wavelength, color="white" if dark else "black",
                          linestyle=(0, (1, 3)), linewidth=2.2, zorder=10)
        if dark:
            line.set_path_effects([pe.Stroke(linewidth=4.5, foreground="black"), pe.Normal()])

def plot_rectangles(ax, data, axis_fit=None, refs=None, title="Spectrum"):
    ax.set_facecolor("#202020")
    for _, row in data.iterrows():
        center = float(row["estimated_nm"])
        # Width is converted using fitted dispersion, if available.
        width_nm = (abs(axis_fit[0])*row["width_px"] if axis_fit else np.nan)
        if not np.isfinite(width_nm) or width_nm <= 0:
            continue
        ax.add_patch(Rectangle((center-width_nm/2, 0.2), width_nm, 0.6,
                               color=wavelength_to_rgb(center), alpha=0.9, zorder=2))
    if refs is not None: add_reference_lines(ax, refs, dark=True)
    ax.set(xlim=(380, 720), ylim=(0, 1), yticks=[], xlabel="Wavelength (nm)", title=title)
    if axis_fit is None:
        ax.text(0.02, 0.04, "Widths unavailable: fewer than 2 calibration anchors",
                transform=ax.transAxes, color="white", fontsize=9)

def plot_intensity(ax, pic, data, axis_fit=None, refs=None, title="Intensity"):
    smooth, peaks = find_peaks_smoothed(vert_sum(pic))
    if axis_fit is None:
        ax.text(0.03, 0.5, "Insufficient anchors for wavelength axis", transform=ax.transAxes)
    else:
        slope, intercept = axis_fit
        wl_axis = slope*np.arange(len(smooth)) + intercept
        ax.plot(wl_axis, np.maximum(smooth, 1e-9), linewidth=1.2)
        ax.scatter(slope*peaks+intercept, np.maximum(smooth[peaks], 1e-9),
                   color="red", s=18, zorder=5)
        for x in peaks:
            ax.annotate(str(int(x)), (slope*x+intercept, max(smooth[x], 1e-9)),
                        xytext=(0, 8), textcoords="offset points", ha="center",
                        fontsize=7, color="red")
    if refs is not None: add_reference_lines(ax, refs)
    ax.set_yscale("log")
    ax.set(xlim=(380, 720), xlabel="Wavelength (nm)", ylabel="RAW intensity (a.u.)", title=title)
    ax.grid(alpha=0.2)

def plot_detected_image(ax, pic, title="Detected peaks on RAW"):
    raw = get_full_pic(str(pic))
    y0, y1 = get_band(pic)
    crop = raw[y0:y1]
    # Display scaling only, never used for the measurements
    low, high = np.percentile(crop, [5, 99.8])
    display_img = np.clip((crop.astype(float)-low)/max(high-low, 1), 0, 1)
    ax.imshow(display_img, cmap="gray", origin="upper", aspect="auto",
              extent=(0, raw.shape[1], y1, y0))
    _, peaks = find_peaks_smoothed(vert_sum(pic))
    for x in peaks:
        ax.axvline(int(x), color="red", linestyle=":", linewidth=1, alpha=0.9)
        ax.text(int(x), y0, str(int(x)), color="red", fontsize=7,
                rotation=90, ha="center", va="bottom")
    ax.set(xlabel="Sensor pixel x", ylabel="Sensor row y", title=title)

def show_spectrum(pic, data, refs=None, reference_map=None, name=None):
    name = name or Path(pic).stem
    axis_fit, source = fit_axis(data, reference_map)
    fig, axes = plt.subplots(3, 1, figsize=(14, 11), constrained_layout=True)
    plot_rectangles(axes[0], data, axis_fit, refs, f"{name} - RGB-estimated bands")
    plot_intensity(axes[1], pic, data, axis_fit, refs,
                   f"{name} - intensity on {source} wavelength axis")
    plot_detected_image(axes[2], pic, f"{name} - detected peaks on RAW image")
    plt.show()
    if axis_fit is not None:
        print(f"{name}: dispersion = {abs(axis_fit[0]):.5f} nm/pixel; axis from {source}.")
    return axis_fit

# Independently assigned peak labels from the original notebook.
helium_bands = {825:388.8, 1285:447.1, 1435:471.3, 1566:492.1,
                1612:501.5, 2136:587.5, 2627:667.8}
hydrogen_lines = {1732:486.0, 2666:656.0}
mercury_lines = {1344:404.7, 1501:435.8, 2081:546.1, 2255:576.9}
reference_lines = {
    "Helium": [388.8,447.1,471.3,492.1,501.5,587.5,667.8,706.5],
    "Hydrogen": [410.17,434.05,486.13,656.28],
    "Mercury": [404.66,435.83,546.07,576.96,579.07]
}
neon_reference_nm = [540.06,585.25,588.19,594.48,597.55,603.00,
    607.43,609.62,614.31,616.36,621.73,626.65,630.48,633.44,
    638.30,640.22,650.65,653.29,659.90,667.83,671.70,692.95,703.24]

def known_path(filename):
    p = KNOWN_DIR / filename
    if not p.exists():
        # Allows the notebook to run beside the supplied example DNGs.
        alternative = Path(filename)
        if alternative.exists(): return alternative
        raise FileNotFoundError(f"Expected {p} (or {alternative})")
    return p


In [ ]:
# 3. Unknown spectrum: put exactly one .dng in data/
files = sorted(DATA_DIR.glob("*.dng"))
if len(files) != 1:
    raise ValueError(f"Expected exactly one .dng in {DATA_DIR}/; found {len(files)}")
unknown_pic = files[0]
unknown = predict(unknown_pic)
if unknown.empty:
    raise ValueError("No valid peaks; inspect exposure, detection threshold and background")
unknown_axis = show_spectrum(unknown_pic, unknown, name=f"Unknown: {unknown_pic.stem}")

# Report both measured pixel widths and approximate wavelength widths.
unknown_export = unknown.copy()
unknown_export["width_nm_approx"] = (unknown_export["width_px"] * abs(unknown_axis[0])
                                      if unknown_axis is not None else np.nan)
unknown_export["width_method"] = ("FWHM-like width at half prominence; "
    "nm conversion from RGB-derived affine dispersion")
outfile = OUTPUT_DIR / f"{unknown_pic.stem}_detected_lines.csv"
unknown_export.to_csv(outfile, index=False)
print(f"Saved {len(unknown_export)} valid detected lines to {outfile}")
display(unknown_export[["pixel", "estimated_nm", "width_px", "width_nm_approx", "colour_distance"]].round(3))


In [ ]:
# 4. Known sources: reference overlays and accuracy for presentation
known_specs = {
    "Helium": (known_path("g6f_he_2_full.dng"), helium_bands),
    "Hydrogen": (known_path("g6f_h_full.dng"), hydrogen_lines),
    "Mercury": (known_path("g6f_hg_full.dng"), mercury_lines)
}
known_results, error_tables = {}, []
for name, (pic, assigned) in known_specs.items():
    data = predict(pic)
    known_results[name] = data
    show_spectrum(pic, data, refs=reference_lines[name],
                  reference_map=assigned, name=name)
    matched = data.copy()
    matched["actual_nm"] = matched["pixel"].map(assigned)
    matched = matched.dropna(subset=["actual_nm"])
    # 388.8 nm helium line is outside the 400-700 nm sensor model.
    matched = matched[matched["actual_nm"].between(WL_MIN, WL_MAX)].copy()
    matched["element"] = name
    matched["error_nm"] = matched["estimated_nm"] - matched["actual_nm"]
    error_tables.append(matched)

errors = pd.concat(error_tables, ignore_index=True)
summary = errors.groupby("element").agg(
    lines=("pixel", "size"),
    MAE_nm=("error_nm", lambda s: s.abs().mean()),
    RMSE_nm=("error_nm", lambda s: np.sqrt(np.mean(s**2))),
    max_abs_error_nm=("error_nm", lambda s: s.abs().max())
).reset_index()
summary.loc[len(summary)] = ["OVERALL", len(errors), errors.error_nm.abs().mean(),
    np.sqrt(np.mean(errors.error_nm**2)), errors.error_nm.abs().max()]
print("Accuracy summary (assigned, in-range lines only)")
display(summary.round(3))
print("Individual errors")
display(errors[["element", "pixel", "actual_nm", "estimated_nm", "error_nm", "width_px", "colour_distance"]].round(3))
fig, ax = plt.subplots(figsize=(9, 4))
for name, group in errors.groupby("element"):
    ax.scatter(group.actual_nm, group.error_nm, label=name, s=45)
ax.axhline(0, color="black", linestyle="--")
ax.set(xlabel="Reference wavelength (nm)", ylabel="Prediction error (nm)",
       title="IMX477 wavelength estimation errors")
ax.legend(); ax.grid(alpha=0.25)
plt.show()
summary.to_csv(OUTPUT_DIR / "known_accuracy_summary.csv", index=False)
errors.to_csv(OUTPUT_DIR / "known_individual_errors.csv", index=False)


In [ ]:
# 5. Suspicious deuterium-labelled tube vs candidate Ne I reference lines
sus_pic = known_path("g6f_deu_full.dng")
deu = predict(sus_pic)
show_spectrum(sus_pic, deu, refs=neon_reference_nm,
              reference_map=None, name="Deuterium-labelled / possible neon")
display(deu[["pixel", "estimated_nm", "width_px", "colour_distance"]].round(3))
print("CAUTION: Neon overlap is exploratory, not a confirmed gas identification.")


In [ ]:
# 6. Example diagnostic: selected helium rows and channel-wise baseline removal
example_pic = known_path("g6f_he_2_full.dng")
raw = get_full_pic(str(example_pic))
y0, y1 = get_band(example_pic)
x, profiles, baseline = get_percentile_baseline(example_pic)
corrected = profiles - baseline

fig, ax = plt.subplots(figsize=(14, 4))
lo, hi = np.percentile(raw, [5, 99.8])
ax.imshow(np.clip((raw.astype(float)-lo)/max(hi-lo,1), 0, 1),
          cmap="gray", origin="upper", aspect="auto")
ax.axhspan(y0, y1, color="cyan", alpha=0.18, label="Rows used")
ax.axhline(y0, color="cyan", linestyle="--")
ax.axhline(y1, color="cyan", linestyle="--")
ax.set(title="Helium RAW: brightest fifth selected for extraction",
       xlabel="Pixel x", ylabel="Pixel y")
ax.legend(); plt.show()

fig, axes = plt.subplots(3, 1, figsize=(14, 9), constrained_layout=True)
for k, color in enumerate(("red", "green", "blue")):
    axes[k].plot(x, profiles[:,k], color=color, alpha=0.65, label="RAW channel")
    axes[k].plot(x, baseline[:,k], "k--", label="Percentile baseline")
    axes[k].plot(x, corrected[:,k], color="gray", alpha=0.65, label="After subtraction")
    axes[k].set(title=f"{color.capitalize()} channel", xlabel="Pixel x",
                ylabel="Mean RAW counts")
    axes[k].legend(); axes[k].grid(alpha=0.2)
plt.show()
print(f"Percentile={PERCENTILE}, window={WINDOW} Bayer blocks (~{2*WINDOW} original pixels)")
print("Inspect whether the dashed baseline follows the background rather than the peaks.")
